# PyTorch 数据集处理：Dataset & DataLoader 快速复习笔记

> **忘记数据怎么喂给 PyTorch 时，打开这份 Notebook 就能快速找到对应场景和模板。**

- Torchvision 自带数据集；
- 按类别文件夹组织的图片；
- 已经在内存里的 Tensor / NumPy；
- CSV / Excel 表格数据；
- CSV 中记录“图片路径 + 标签”；
- 图片与标签文件分开放；
- 图像分割：image + mask；
- 目标检测：image + boxes / labels；
- 文本、时间序列等变长样本；
- 超大文件 / 流式数据；
- 训练集、验证集、测试集划分；
- 类别不平衡采样；
- `collate_fn`；
- `DataLoader` 性能参数；
- GPU 数据搬运；
- 常见错误与项目模板。

---

## 整体框架

```text
原始数据
   ↓
Dataset：规定“第 i 个样本怎么取”
   ↓
Sampler：决定“按什么顺序取样本”（可选）
   ↓
DataLoader：把多个样本组装成 batch，并负责打乱、多进程加载等
   ↓
batch = (X, y)
   ↓
model(X)
```

核心理解：

- **Dataset 管单个样本**
- **DataLoader 管一个 batch**

## 0. “场景 → 写法”速查表

| 你的数据是什么 | 推荐方式 | 是否需要自己写 `Dataset` |
|---|---|---|
| FashionMNIST / CIFAR10 等官方数据集 | `torchvision.datasets.xxx` | 否 |
| 图片按类别放在不同文件夹 | `datasets.ImageFolder` | 否 |
| 特征和标签已经是 Tensor | `TensorDataset(X, y)` | 否 |
| NumPy 数组 | `torch.from_numpy` → `TensorDataset` | 通常否 |
| CSV/Excel 纯表格特征 | pandas → Tensor → `TensorDataset` | 通常否 |
| CSV 保存图片路径 + 标签 | 自定义 `Dataset` | 是 |
| 图片目录和标签文件分离 | 自定义 `Dataset` | 是 |
| 分割：图片 + mask | 自定义 `Dataset` | 是 |
| 检测：图片 + boxes/labels | 自定义 `Dataset` | 是 |
| 文本/时间序列长度不一致 | `Dataset` + 自定义 `collate_fn` | 经常是 |
| 数据太大、无法随机索引 | `IterableDataset` | 经常是 |
| 多个已有 Dataset 合并 | `ConcatDataset` | 否 |
| 从已有 Dataset 取一部分 | `Subset` | 否 |

### 最实用的判断原则

如果数据已经能自然表示成：

```python
dataset[i] -> (x_i, y_i)
```

并且 PyTorch 已经有现成类，就直接用现成类。

如果你需要自己规定：

> “给我索引 `i`，我要去哪里读文件、怎么预处理、返回什么标签？”

那就写自定义 `Dataset`。

In [ ]:
# 1. 常用导入：以后大多数数据处理 Notebook 都可以从这里开始

from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch.utils.data import (
    Dataset,
    IterableDataset,
    DataLoader,
    TensorDataset,
    Subset,
    ConcatDataset,
    random_split,
    WeightedRandomSampler,
)
from torchvision import datasets
from torchvision.transforms import v2

# 1. Dataset 和 DataLoader 到底分别干什么？

## Dataset

`Dataset` 可以理解成：

> **一个“可按索引取样本”的数据集合。**

最重要的接口：

```python
len(dataset)      # 有多少个样本
dataset[i]        # 第 i 个样本
```

对于监督学习，一般：

```python
dataset[i] -> (x, y)
```

例如：

```python
image, label = train_dataset[0]
```

---

## DataLoader

`DataLoader` 是在 Dataset 外面再套一层：

```python
loader = DataLoader(
    dataset,
    batch_size=32,
    shuffle=True
)
```

于是：

```python
for X, y in loader:
    ...
```

拿到的不再是一个样本，而通常是：

```text
X.shape = [batch_size, ...]
y.shape = [batch_size, ...]
```

### 记忆

```text
Dataset[0]
↓
一个样本

DataLoader
↓
一批样本（batch）
```

In [ ]:
# 用最简单的 TensorDataset 演示 Dataset → DataLoader

X_demo = torch.randn(10, 4)
y_demo = torch.randint(0, 3, (10,))

demo_dataset = TensorDataset(X_demo, y_demo)
demo_loader = DataLoader(demo_dataset, batch_size=4, shuffle=True)

print("数据集长度：", len(demo_dataset))
print("单个样本：")
print(demo_dataset[0])

X_batch, y_batch = next(iter(demo_loader))
print("\n一个 batch：")
print("X_batch.shape =", X_batch.shape)
print("y_batch.shape =", y_batch.shape)

# 2. 情景一：PyTorch / Torchvision 自带 Dataset

你之前使用的 `FashionMNIST` 就属于这种情况。

典型结构：

```python
dataset = datasets.FashionMNIST(
    root="data",
    train=True,
    download=True,
    transform=...
)
```

常见参数：

| 参数 | 作用 |
|---|---|
| `root` | 数据存放位置 |
| `train=True/False` | 训练集还是测试集（部分数据集用 `split=`） |
| `download=True` | 本地没有时自动下载 |
| `transform` | 对输入 `x` 做处理 |
| `target_transform` | 对标签 `y` 做处理 |

当前 Torchvision 更推荐 `transforms.v2`。一个常见图像转换流程：

```python
v2.ToImage()
v2.ToDtype(torch.float32, scale=True)
```

把 PIL / NumPy 图像转换为 Tensor，并得到常用的 float 图像表示。

In [ ]:
# 示例：FashionMNIST
# 第一次运行会联网下载，因此这里仅作为模板。

fashion_transform = v2.Compose([
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True),
])

# train_dataset = datasets.FashionMNIST(
#     root="data",
#     train=True,
#     download=True,
#     transform=fashion_transform,
# )
#
# test_dataset = datasets.FashionMNIST(
#     root="data",
#     train=False,
#     download=True,
#     transform=fashion_transform,
# )
#
# train_loader = DataLoader(
#     train_dataset,
#     batch_size=64,
#     shuffle=True,
# )
#
# test_loader = DataLoader(
#     test_dataset,
#     batch_size=64,
#     shuffle=False,
# )

## 常见 Torchvision 数据集

图像分类中常见：

```python
datasets.MNIST
datasets.FashionMNIST
datasets.CIFAR10
datasets.CIFAR100
datasets.ImageNet
datasets.Food101
datasets.Flowers102
...
```

Torchvision 还提供检测、分割、视频等数据集。

### 核心规律

这些类本身已经是 `Dataset`：

```text
torchvision.datasets.FashionMNIST
                    ↓
                 Dataset
                    ↓
               DataLoader
```

所以不需要再自己继承 `Dataset`。

# 3. 情景二：图片已经按“类别文件夹”整理 —— `ImageFolder`

这是你刚刚遇到的场景。

目录必须类似：

```text
data/
├── train/
│   ├── cat/
│   │   ├── 001.jpg
│   │   └── 002.jpg
│   └── dog/
│       ├── 001.jpg
│       └── 002.jpg
│
└── test/
    ├── cat/
    └── dog/
```

规则：

> **一个子文件夹 = 一个类别**

于是直接：

```python
datasets.ImageFolder("data/train")
```

即可。

`ImageFolder` 会自动：

1. 扫描类别文件夹；
2. 建立类别名；
3. 把类别映射成整数；
4. 保存图片路径和对应标签；
5. 在 `dataset[i]` 时读取图片。

In [ ]:
# ImageFolder 标准模板

data_dir = Path("data")

train_transform = v2.Compose([
    v2.ToImage(),
    v2.RandomResizedCrop((224, 224)),
    v2.RandomHorizontalFlip(p=0.5),
    v2.ToDtype(torch.float32, scale=True),
])

test_transform = v2.Compose([
    v2.ToImage(),
    v2.Resize((224, 224)),
    v2.ToDtype(torch.float32, scale=True),
])

# train_dataset = datasets.ImageFolder(
#     root=data_dir / "train",
#     transform=train_transform,
# )
#
# test_dataset = datasets.ImageFolder(
#     root=data_dir / "test",
#     transform=test_transform,
# )
#
# print(train_dataset.classes)
# print(train_dataset.class_to_idx)
#
# train_loader = DataLoader(
#     train_dataset,
#     batch_size=32,
#     shuffle=True,
# )
#
# test_loader = DataLoader(
#     test_dataset,
#     batch_size=32,
#     shuffle=False,
# )

## `ImageFolder` 常用属性

```python
dataset.classes
```

例如：

```python
['cat', 'dog']
```

---

```python
dataset.class_to_idx
```

例如：

```python
{'cat': 0, 'dog': 1}
```

---

```python
dataset.samples
```

大致是：

```python
[
    ('data/train/cat/001.jpg', 0),
    ('data/train/cat/002.jpg', 0),
    ('data/train/dog/001.jpg', 1),
]
```

### 注意

类别编号由 `ImageFolder` 根据类别目录建立。

**不要想当然地认为某个类别一定是 0 或 1。**

项目里直接检查：

```python
print(dataset.class_to_idx)
```

# 4. 情景三：数据已经是 Tensor —— `TensorDataset`

这是最简单的自有数据形式。

假设：

```python
X.shape = [N, D]
y.shape = [N]
```

直接：

```python
dataset = TensorDataset(X, y)
```

`TensorDataset` 的本质就是：

```python
dataset[i] -> (X[i], y[i])
```

适用于：

- 人工生成的数据；
- 已提前处理好的特征；
- MLP 输入；
- 表格数据转 Tensor 后；
- 已经提取好的 embedding；
- 已经离线提取好的 CNN / Transformer 特征。

In [ ]:
# TensorDataset：最重要的模板之一

N = 100
X = torch.randn(N, 20)
y = torch.randint(0, 2, (N,))

dataset = TensorDataset(X, y)

train_loader = DataLoader(
    dataset,
    batch_size=16,
    shuffle=True,
)

X_batch, y_batch = next(iter(train_loader))

print(X_batch.shape)
print(y_batch.shape)

## `TensorDataset` 不只支持两个 Tensor

例如模型需要：

```text
input_ids
attention_mask
labels
```

也可以：

```python
dataset = TensorDataset(
    input_ids,
    attention_mask,
    labels
)
```

取样时：

```python
input_ids_i, mask_i, label_i = dataset[i]
```

前提：

> 所有 Tensor 的第 0 维样本数量必须一致。

In [ ]:
# 一个样本包含多个 Tensor

input_ids = torch.randint(0, 1000, (100, 32))
attention_mask = torch.ones(100, 32, dtype=torch.long)
labels = torch.randint(0, 2, (100,))

nlp_tensor_dataset = TensorDataset(
    input_ids,
    attention_mask,
    labels,
)

batch = next(iter(DataLoader(nlp_tensor_dataset, batch_size=8)))

print([x.shape for x in batch])

# 5. 情景四：NumPy 数组 → Dataset

如果你现在有：

```python
X_numpy
y_numpy
```

通常先转 Tensor：

```python
X = torch.from_numpy(X_numpy)
y = torch.from_numpy(y_numpy)
```

再：

```python
TensorDataset(X, y)
```

### `torch.from_numpy()` 和 `torch.tensor()` 的区别

常见情况下：

```python
torch.from_numpy(arr)
```

会与 NumPy 数组共享底层内存，更高效。

而：

```python
torch.tensor(arr)
```

通常会复制数据。

深度学习训练时还要注意 dtype：

- 输入特征通常 `float32`
- 多分类标签通常 `long`

In [ ]:
# NumPy → TensorDataset

X_np = np.random.randn(100, 10).astype(np.float32)
y_np = np.random.randint(0, 3, size=100).astype(np.int64)

X = torch.from_numpy(X_np)
y = torch.from_numpy(y_np)

dataset = TensorDataset(X, y)
loader = DataLoader(dataset, batch_size=16, shuffle=True)

X_batch, y_batch = next(iter(loader))

print(X_batch.dtype)  # 常见：torch.float32
print(y_batch.dtype)  # CrossEntropyLoss 常见要求：torch.int64 / long

# 6. 情景五：CSV / Excel 表格数据

例如：

```text
age,height,weight,label
21,175,65,0
30,180,80,1
...
```

常见流程：

```text
CSV / Excel
   ↓
pandas.DataFrame
   ↓
缺失值 / 类别编码 / 标准化
   ↓
NumPy
   ↓
Tensor
   ↓
TensorDataset
   ↓
DataLoader
```

如果数据已经可以一次性放进内存，**通常没有必要专门写自定义 Dataset**。

In [ ]:
# CSV 表格数据标准模板
# 假设最后一列叫 label

# df = pd.read_csv("data/train.csv")
#
# X_np = df.drop(columns=["label"]).to_numpy(dtype=np.float32)
# y_np = df["label"].to_numpy(dtype=np.int64)
#
# X = torch.from_numpy(X_np)
# y = torch.from_numpy(y_np)
#
# dataset = TensorDataset(X, y)
# loader = DataLoader(
#     dataset,
#     batch_size=64,
#     shuffle=True,
# )

## 表格数据最容易犯的一个错误：数据泄漏

假设要做标准化：

```python
x = (x - mean) / std
```

不要：

```text
全部数据
↓
先算 mean/std
↓
再划 train/val/test
```

因为测试集信息泄漏到了训练阶段。

正确：

```text
全部数据
↓
先划 train / val / test
↓
只用 train 计算 mean/std
↓
把同一组 mean/std 用到 val/test
```

这个原则不仅适用于标准化，也适用于：

- 缺失值统计；
- 特征选择；
- PCA；
- 类别编码中依赖数据统计的部分；
- 任何“从数据里学习参数”的预处理。

# 7. 情景六：CSV 里保存“图片路径 + 标签” —— 自定义 Dataset

真实项目中非常常见：

```text
filename,label
0001.jpg,cat
0002.jpg,dog
0003.jpg,cat
```

此时不能直接用 `ImageFolder`，因为类别信息不一定由文件夹表达。

最通用的方法：

```python
class MyDataset(Dataset):
    def __init__(...):
        ...

    def __len__(self):
        ...

    def __getitem__(self, idx):
        ...
```

牢记三个职责：

### `__init__`

保存：

- 路径；
- DataFrame；
- transform；
- 类别映射。

### `__len__`

告诉 PyTorch：

> 一共有多少个样本。

### `__getitem__(idx)`

告诉 PyTorch：

> 第 `idx` 个样本到底怎么读取和返回。

In [ ]:
from PIL import Image

class ImageCSVDataset(Dataset):
    def __init__(self, csv_file, image_dir, transform=None):
        self.df = pd.read_csv(csv_file)
        self.image_dir = Path(image_dir)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        image_path = self.image_dir / row["filename"]
        image = Image.open(image_path).convert("RGB")

        label = int(row["label"])

        if self.transform is not None:
            image = self.transform(image)

        return image, label


# 使用方式：
#
# transform = v2.Compose([
#     v2.ToImage(),
#     v2.Resize((224, 224)),
#     v2.ToDtype(torch.float32, scale=True),
# ])
#
# dataset = ImageCSVDataset(
#     csv_file="data/train.csv",
#     image_dir="data/images",
#     transform=transform,
# )
#
# loader = DataLoader(
#     dataset,
#     batch_size=32,
#     shuffle=True,
# )

## 自定义 Dataset 的万能模板

以后不知道怎么写时，就从这里复制：

```python
class MyDataset(Dataset):

    def __init__(self, ...):
        # 保存路径、标签、transform 等
        ...

    def __len__(self):
        return 数据集长度

    def __getitem__(self, idx):
        # 1. 根据 idx 找到第 idx 个样本
        ...

        # 2. 读取数据
        ...

        # 3. 做必要预处理
        ...

        # 4. 返回
        return x, y
```

### 一个特别重要的设计原则

通常不要在 `__init__` 里一次性把几万张大图全部读进 RAM。

更常见的做法：

```text
__init__
↓
只记录文件路径

__getitem__
↓
需要第 i 张时再读第 i 张
```

这叫 **lazy loading（惰性加载）**。

# 8. 情景七：图片在一个目录，标签在另一个文件 / Python 列表

比如：

```text
images/
├── a.jpg
├── b.jpg
└── c.jpg

labels.txt
```

或者你手里已经有：

```python
image_paths = [...]
labels = [...]
```

直接写自定义 Dataset。

In [ ]:
class PathLabelDataset(Dataset):
    def __init__(self, image_paths, labels, transform=None):
        assert len(image_paths) == len(labels)

        self.image_paths = [Path(p) for p in image_paths]
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        image = Image.open(self.image_paths[idx]).convert("RGB")
        label = self.labels[idx]

        if self.transform is not None:
            image = self.transform(image)

        return image, label

# 9. 情景八：图像分割 —— image + mask

语义分割的一个样本不是：

```python
(image, class_label)
```

而是：

```python
(image, mask)
```

例如：

```text
images/
├── 001.jpg
└── 002.jpg

masks/
├── 001.png
└── 002.png
```

最关键的问题：

> **随机裁剪、水平翻转、旋转等几何增强必须让 image 和 mask 完全同步。**

不能：

```python
image = random_crop(image)
mask = random_crop(mask)
```

因为两次随机裁剪可能位置不同。

现代 `torchvision.transforms.v2` 的一个重要优势就是可以对图像、mask、框等结构化输入做同步变换。

In [ ]:
from torchvision import tv_tensors

class SegmentationDataset(Dataset):
    def __init__(self, image_paths, mask_paths, transforms=None):
        assert len(image_paths) == len(mask_paths)

        self.image_paths = [Path(p) for p in image_paths]
        self.mask_paths = [Path(p) for p in mask_paths]
        self.transforms = transforms

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        image = Image.open(self.image_paths[idx]).convert("RGB")
        mask = Image.open(self.mask_paths[idx])

        image = v2.ToImage()(image)

        # Mask 保留类别编号语义，不像普通 RGB 图片那样缩放到 [0, 1]
        mask = torch.as_tensor(np.array(mask), dtype=torch.long)
        mask = tv_tensors.Mask(mask)

        if self.transforms is not None:
            image, mask = self.transforms(image, mask)

        return image, mask


seg_train_transform = v2.Compose([
    v2.RandomHorizontalFlip(p=0.5),
    v2.RandomResizedCrop((256, 256)),
    v2.ToDtype(torch.float32, scale=True),
])

## 为什么 mask 不能简单当普通图片处理？

假设 mask 像素：

```text
0 = 背景
1 = 猫
2 = 狗
```

它存的是**类别编号**，不是普通颜色强度。

因此不能随便：

- 把 `2` 变成 `2/255`；
- 用会产生中间数值的插值方式破坏类别编号。

这是分割数据处理里非常重要的概念。

# 10. 情景九：目标检测 —— image + target 字典

检测任务的一个样本通常类似：

```python
image, target
```

其中：

```python
target = {
    "boxes": ...,
    "labels": ...,
}
```

例如：

```text
boxes.shape  = [目标数量, 4]
labels.shape = [目标数量]
```

因为每张图片里的目标数量不同，所以检测任务也经常需要特别的 batch 组织方式。

In [ ]:
class DetectionDataset(Dataset):
    def __init__(self, records, transforms=None):
        self.records = records
        self.transforms = transforms

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        record = self.records[idx]

        image = Image.open(record["image_path"]).convert("RGB")
        image = v2.ToImage()(image)

        boxes = torch.tensor(record["boxes"], dtype=torch.float32)
        labels = torch.tensor(record["labels"], dtype=torch.int64)

        boxes = tv_tensors.BoundingBoxes(
            boxes,
            format="XYXY",
            canvas_size=image.shape[-2:],
        )

        target = {
            "boxes": boxes,
            "labels": labels,
        }

        if self.transforms is not None:
            image, target = self.transforms(image, target)

        image = v2.ToDtype(torch.float32, scale=True)(image)

        return image, target

## 检测任务为什么默认 batch 可能失败？

分类任务：

```text
image1: [3,224,224], label: 1
image2: [3,224,224], label: 0
```

很容易 stack：

```text
images: [2,3,224,224]
labels: [2]
```

但是检测：

```text
第 1 张：3 个框
第 2 张：7 个框
```

框数量不同，不能直接：

```python
torch.stack([boxes1, boxes2])
```

因此经常自定义：

```python
collate_fn
```

In [ ]:
# 检测任务常见 collate_fn

def detection_collate_fn(batch):
    images, targets = zip(*batch)
    return list(images), list(targets)


# loader = DataLoader(
#     detection_dataset,
#     batch_size=4,
#     shuffle=True,
#     collate_fn=detection_collate_fn,
# )

# 11. `collate_fn` 到底是什么？

`Dataset` 返回的是**一个样本**：

```python
dataset[i]
```

例如：

```python
(x_i, y_i)
```

当 `batch_size=4` 时，DataLoader 会先拿到：

```python
[
    dataset[i1],
    dataset[i2],
    dataset[i3],
    dataset[i4],
]
```

然后：

```python
collate_fn(...)
```

负责：

> **如何把这 4 个样本拼成一个 batch。**

默认 `collate_fn` 会自动把规则 Tensor stack 起来。

所以普通分类通常根本不用管它。

只有当样本形状无法直接堆叠，或者你需要特殊 batch 结构时，才自己写。

# 12. 情景十：文本 / 时间序列长度不同 —— 自定义 `collate_fn`

例如：

```text
句子 A：长度 5
句子 B：长度 8
句子 C：长度 3
```

默认：

```python
torch.stack(...)
```

无法直接堆起来。

常见方法：

```text
Dataset
↓
返回未 padding 的序列
↓
collate_fn
↓
在当前 batch 内统一 padding
```

这样比“整个数据集都 padding 到全局最大长度”通常更节省计算。

In [ ]:
from torch.nn.utils.rnn import pad_sequence

class SequenceDataset(Dataset):
    def __init__(self, sequences, labels):
        self.sequences = sequences
        self.labels = labels

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        sequence = torch.tensor(self.sequences[idx], dtype=torch.long)
        label = torch.tensor(self.labels[idx], dtype=torch.long)

        return sequence, label


def sequence_collate_fn(batch):
    sequences, labels = zip(*batch)

    # [不同长度 Tensor] -> [batch, 当前 batch 最大长度]
    padded_sequences = pad_sequence(
        sequences,
        batch_first=True,
        padding_value=0,
    )

    labels = torch.stack(labels)

    return padded_sequences, labels


sequences = [
    [1, 2, 3],
    [4, 5, 6, 7, 8],
    [9, 10],
    [11, 12, 13, 14],
]
labels = [0, 1, 0, 1]

sequence_dataset = SequenceDataset(sequences, labels)
sequence_loader = DataLoader(
    sequence_dataset,
    batch_size=3,
    shuffle=False,
    collate_fn=sequence_collate_fn,
)

X_batch, y_batch = next(iter(sequence_loader))
print(X_batch)
print("shape =", X_batch.shape)

# 13. 情景十一：Dataset 返回字典

Dataset 不一定必须返回：

```python
(x, y)
```

也可以返回：

```python
{
    "image": image,
    "label": label,
    "filename": filename,
}
```

默认 DataLoader 对规则结构的字典也能很好地组 batch。

In [ ]:
class DictDataset(Dataset):
    def __init__(self, X, y):
        self.X = X
        self.y = y

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return {
            "features": self.X[idx],
            "label": self.y[idx],
        }


dict_dataset = DictDataset(
    torch.randn(12, 5),
    torch.randint(0, 2, (12,))
)

dict_loader = DataLoader(dict_dataset, batch_size=4)

batch = next(iter(dict_loader))

print(batch.keys())
print(batch["features"].shape)
print(batch["label"].shape)

# 14. 情景十二：数据太大 / 流式读取 —— `IterableDataset`

普通 `Dataset` 属于 **map-style dataset（映射式数据集）**：

```python
dataset[i]
```

意味着：

> 我可以根据索引随机拿第 i 条数据。

但是有些数据不适合：

- 几百 GB / TB；
- 日志流；
- 网络流；
- 无限数据生成器；
- 一个超大文本文件逐行读取；
- 数据只能顺序读取。

此时可以使用：

```python
IterableDataset
```

核心接口：

```python
__iter__()
```

而不是重点依赖：

```python
__getitem__()
```

In [ ]:
class NumberStreamDataset(IterableDataset):
    def __init__(self, start, end):
        self.start = start
        self.end = end

    def __iter__(self):
        for i in range(self.start, self.end):
            x = torch.tensor([i], dtype=torch.float32)
            y = torch.tensor(i % 2, dtype=torch.long)
            yield x, y


stream_dataset = NumberStreamDataset(0, 10)
stream_loader = DataLoader(stream_dataset, batch_size=4)

for X_batch, y_batch in stream_loader:
    print(X_batch.squeeze(), y_batch)

## `Dataset` vs `IterableDataset`

| | `Dataset` | `IterableDataset` |
|---|---|---|
| 核心 | `__getitem__` | `__iter__` |
| 能否按索引取 | 通常可以 | 通常不强调 |
| `shuffle=True` | 常用 | 不能简单按普通方式理解 |
| 典型场景 | 普通训练数据 | 流式/超大数据 |
| 例子 | 图片、表格 | 日志流、超大文本流 |

### 多进程注意

`IterableDataset + num_workers > 0` 时，每个 worker 都可能拥有自己的 Dataset 副本。

如果自己写流式数据集，要小心：

> 不要让多个 worker 把同一批数据重复读取一遍。

这时常使用：

```python
torch.utils.data.get_worker_info()
```

给不同 worker 划分不同数据范围。

In [ ]:
# IterableDataset 多 worker 划分模板

class WorkerAwareRangeDataset(IterableDataset):
    def __init__(self, start, end):
        self.start = start
        self.end = end

    def __iter__(self):
        worker_info = torch.utils.data.get_worker_info()

        if worker_info is None:
            # 单进程
            start = self.start
            end = self.end

        else:
            # 每个 worker 负责一段不同的范围
            total = self.end - self.start
            per_worker = (total + worker_info.num_workers - 1) // worker_info.num_workers

            start = self.start + worker_info.id * per_worker
            end = min(start + per_worker, self.end)

        for i in range(start, end):
            yield i

# 15. 训练集 / 验证集怎么划分？—— `random_split`

如果已有：

```python
full_dataset
```

可以：

```python
train_dataset, val_dataset = random_split(...)
```

### 推荐固定随机种子

否则每次运行划分结果可能变化。

In [ ]:
# random_split 可复现模板

full_dataset = TensorDataset(
    torch.randn(100, 10),
    torch.randint(0, 2, (100,))
)

generator = torch.Generator().manual_seed(42)

train_dataset, val_dataset = random_split(
    full_dataset,
    lengths=[80, 20],
    generator=generator,
)

print(len(train_dataset), len(val_dataset))

## 非常重要：先划分 Dataset，再决定训练增强与验证增强

对于图片分类，经常需要：

```text
train → 随机增强
val   → 不做随机增强
test  → 不做随机增强
```

如果原始 Dataset 内已经绑定了随机 `transform`，然后简单 `random_split`：

```python
train_subset, val_subset = random_split(dataset, ...)
```

两个 Subset 实际上仍然引用同一个原 Dataset，因此 transform 也相同。

### 更稳妥的方式

真实图片项目中常直接：

```text
train/
val/
test/
```

分别构造三个 Dataset。

或者：

- 自己维护索引；
- 为训练、验证分别构造 Dataset；
- 再通过 `Subset` 使用对应索引。

# 16. `Subset`：从已有 Dataset 中取一部分

如果已经知道样本索引：

```python
indices = [0, 2, 5, 8, ...]
```

可以：

```python
subset = Subset(dataset, indices)
```

它不会复制所有原始数据，而是记录：

```text
原 Dataset
+
我要哪些索引
```

In [ ]:
base_dataset = TensorDataset(
    torch.arange(20).float().unsqueeze(1),
    torch.arange(20) % 2,
)

indices = [0, 3, 5, 10]
small_subset = Subset(base_dataset, indices)

for sample in small_subset:
    print(sample)

# 17. `ConcatDataset`：多个 Dataset 合并

例如：

```python
dataset_a
dataset_b
dataset_c
```

可以：

```python
full_dataset = ConcatDataset([
    dataset_a,
    dataset_b,
    dataset_c
])
```

常见场景：

- 多个数据来源；
- 多个文件夹；
- 把多个已有 Dataset 拼起来；
- 跨域数据训练。

前提是：

> 它们返回的样本结构最好保持一致。

In [ ]:
dataset_a = TensorDataset(
    torch.randn(10, 4),
    torch.zeros(10, dtype=torch.long),
)

dataset_b = TensorDataset(
    torch.randn(8, 4),
    torch.ones(8, dtype=torch.long),
)

combined_dataset = ConcatDataset([dataset_a, dataset_b])

print(len(combined_dataset))

# 18. 类别不平衡：`WeightedRandomSampler`

假设：

```text
类别 0：950 张
类别 1： 50 张
```

如果正常 shuffle：

```python
shuffle=True
```

小类别仍然很少。

一种处理方式：

```python
WeightedRandomSampler
```

思想：

> 给少数类样本更大的被抽中概率。

注意：

- 它改变的是**采样概率**；
- 它不是 loss 的 class weight；
- `sampler=` 和 `shuffle=True` 通常不要同时使用。

In [ ]:
# WeightedRandomSampler 演示

labels = torch.tensor(
    [0] * 90 + [1] * 10,
    dtype=torch.long
)

X = torch.randn(100, 5)
dataset = TensorDataset(X, labels)

class_counts = torch.bincount(labels)
class_weights = 1.0 / class_counts.float()

sample_weights = class_weights[labels]

sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(sample_weights),
    replacement=True,
)

balanced_loader = DataLoader(
    dataset,
    batch_size=16,
    sampler=sampler,
    # shuffle=True,   # 不要再同时使用
)

sampled_labels = []
for _, y_batch in balanced_loader:
    sampled_labels.extend(y_batch.tolist())

print("原始类别数量：", torch.bincount(labels))
print("本轮采样类别数量：", torch.bincount(torch.tensor(sampled_labels)))

# 19. DataLoader 最常用参数：必须熟悉

标准模板：

```python
loader = DataLoader(
    dataset,
    batch_size=32,
    shuffle=True,
    num_workers=4,
    pin_memory=True,
    drop_last=False,
)
```

---

## `batch_size`

```python
batch_size=32
```

每次取 32 个样本。

---

## `shuffle`

训练：

```python
shuffle=True
```

一般每个 epoch 重新打乱样本顺序。

验证 / 测试：

```python
shuffle=False
```

通常没必要打乱。

---

## `num_workers`

```python
num_workers=0
```

主进程负责读取数据。

```python
num_workers=4
```

使用多个 worker 并行准备数据。

**不是越大越好。**

实际要根据：

- CPU；
- 磁盘；
- 数据增强复杂度；
- 操作系统；
- RAM；
- GPU 速度；

进行测试。

Jupyter / Windows 下调试 Dataset 时，先用：

```python
num_workers=0
```

通常最方便定位报错。

---

## `pin_memory`

GPU 训练时常尝试：

```python
pin_memory=True
```

配合：

```python
X = X.to(device, non_blocking=True)
```

有机会提高 CPU → GPU 数据传输效率。

---

## `drop_last`

假设：

```text
103 个样本
batch_size = 32
```

得到：

```text
32
32
32
7
```

如果：

```python
drop_last=True
```

最后 7 个样本的 batch 被丢掉。

训练某些依赖稳定 batch 统计的场景可能有用。

验证 / 测试通常：

```python
drop_last=False
```

避免漏样本。

# 20. DataLoader 进阶性能参数

这些不用一开始全部背，但忘记时可以查这里。

## `persistent_workers=True`

当：

```python
num_workers > 0
```

时，让 worker 在一个 epoch 结束后继续存在，而不是每个 epoch 都重新创建。

数据加载启动成本较高时可能有帮助。

---

## `prefetch_factor`

控制每个 worker 提前准备多少个 batch。

适合数据读取 / transform 比较慢的场景。

---

## `pin_memory=True`

CUDA 训练时常值得测试。

---

## `num_workers`

性能调优时最值得实际 benchmark 的参数之一。

不要机械地写：

```python
num_workers=os.cpu_count()
```

因为最优值与机器和任务有关。

### 性能调优思路

如果训练时 GPU 利用率经常：

```text
高
↓
掉到很低
↓
再高
```

其中一种可能是：

> GPU 在等 DataLoader 准备下一批数据。

这时可检查：

- `num_workers`；
- 图片解码；
- transform 是否太重；
- `pin_memory`；
- 存储设备 I/O；
- batch size。

In [ ]:
# 一个偏实战的 DataLoader 模板

def make_loader(
    dataset,
    batch_size=64,
    train=True,
    num_workers=0,
    use_cuda=False,
):
    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=train,
        num_workers=num_workers,
        pin_memory=use_cuda,
        drop_last=False,
        persistent_workers=(num_workers > 0),
    )

# 21. Batch 怎么搬到 GPU？

Dataset / DataLoader 通常负责在 CPU 侧准备数据。

训练循环：

```python
for X, y in train_loader:
    X = X.to(device)
    y = y.to(device)
```

如果使用：

```python
pin_memory=True
```

常见搭配：

```python
X = X.to(device, non_blocking=True)
```

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 标准训练循环的数据部分
#
# for X, y in train_loader:
#     X = X.to(device, non_blocking=True)
#     y = y.to(device, non_blocking=True)
#
#     logits = model(X)
#     loss = criterion(logits, y)
#     ...

# 22. 图像 Transform 应该放在哪里？

最常见：

```python
dataset = ImageFolder(
    ...,
    transform=train_transform
)
```

或者自定义 Dataset：

```python
if self.transform is not None:
    image = self.transform(image)
```

这样 transform 在：

```python
dataset[i]
```

取样本时执行。

如果 transform 是随机增强，那么**同一张原图在不同 epoch 被读取时可能得到不同结果**。

这正是数据增强想要的效果。

---

## 训练和测试 Transform 分开

训练：

```python
RandomResizedCrop
RandomHorizontalFlip
ColorJitter
...
```

测试：

```python
Resize
CenterCrop
...
```

测试阶段一般不要做随机增强。

In [ ]:
# 分类任务：一个较标准的 train / test transform 示例

train_transform = v2.Compose([
    v2.ToImage(),
    v2.RandomResizedCrop((224, 224)),
    v2.RandomHorizontalFlip(p=0.5),
    v2.ColorJitter(
        brightness=0.2,
        contrast=0.2,
        saturation=0.2,
        hue=0.1,
    ),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225],
    ),
])

test_transform = v2.Compose([
    v2.ToImage(),
    v2.Resize((256, 256)),
    v2.CenterCrop((224, 224)),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225],
    ),
])

## 关于你之前学过的 `ToTensor()`

老代码中非常常见：

```python
from torchvision import transforms

transforms.ToTensor()
```

现在官方更推荐 v2 风格：

```python
from torchvision.transforms import v2

v2.ToImage()
v2.ToDtype(torch.float32, scale=True)
```

所以复习旧项目时看到 `ToTensor()` 不需要觉得它“写错了”。

你只需要知道：

```text
旧项目 / 教程
→ transforms.ToTensor()

现代 torchvision v2
→ ToImage() + ToDtype(..., scale=True)
```

# 23. 一个 Dataset 到底返回什么 dtype？

分类任务非常常见：

```python
image.dtype == torch.float32
label.dtype == torch.long
```

例如使用：

```python
nn.CrossEntropyLoss()
```

标签通常应该是整数类别索引：

```text
0
1
2
...
C-1
```

而不是：

```text
0.0
1.0
2.0
```

---

## 二分类 BCE 类型任务

如果最后输出一个 logit，并使用：

```python
nn.BCEWithLogitsLoss()
```

标签通常会使用浮点：

```python
0.0
1.0
```

因此：

> **标签 dtype 不是永远固定的，要看损失函数和任务定义。**

# 24. 调试 Dataset：永远先做这几步

不要一上来就训练 100 个 epoch。

先检查：

```python
print(len(dataset))
```

再：

```python
x, y = dataset[0]
```

检查：

```python
type(x)
x.shape
x.dtype
y
```

然后：

```python
batch = next(iter(loader))
```

再检查 batch。

### 推荐调试顺序

```text
1. dataset 能不能创建？
        ↓
2. len(dataset) 对不对？
        ↓
3. dataset[0] 对不对？
        ↓
4. DataLoader 能不能取第一个 batch？
        ↓
5. batch shape / dtype 对不对？
        ↓
6. 才送进模型
```

这能快速区分：

```text
数据读取问题
vs
模型问题
```

In [ ]:
def inspect_classification_dataset(dataset, loader=None):
    print("dataset type:", type(dataset).__name__)
    print("dataset length:", len(dataset))

    x, y = dataset[0]

    print("\n--- 单个样本 ---")
    print("x type:", type(x))
    if hasattr(x, "shape"):
        print("x shape:", x.shape)
    if hasattr(x, "dtype"):
        print("x dtype:", x.dtype)

    print("y:", y)
    if hasattr(y, "dtype"):
        print("y dtype:", y.dtype)

    if loader is not None:
        X, Y = next(iter(loader))

        print("\n--- 一个 batch ---")
        print("X shape:", X.shape)
        print("X dtype:", X.dtype)
        print("Y shape:", Y.shape)
        print("Y dtype:", Y.dtype)

# 25. 常见错误 1：`ImageFolder` 目录结构放错

错误：

```text
train/
├── 001_cat.jpg
├── 002_dog.jpg
└── ...
```

然后：

```python
ImageFolder("train")
```

这种结构没有类别子目录，因此不适合直接使用 ImageFolder。

### 两种解决方式

#### 方法 A：整理文件夹

```text
train/
├── cat/
└── dog/
```

#### 方法 B：自定义 Dataset

自己从：

```text
文件名 / CSV / JSON / 标签文件
```

读取标签。

# 26. 常见错误 2：Dataset 返回的图片尺寸不同

例如：

```text
image1 = [3, 400, 300]
image2 = [3, 512, 512]
```

默认 DataLoader 无法直接组成：

```text
[batch, C, H, W]
```

分类任务一般先：

```python
Resize
RandomResizedCrop
CenterCrop
```

统一尺寸。

如果任务本身必须保留不同尺寸：

- 需要特殊 `collate_fn`；
- 或按任务框架要求组织 batch。

# 27. 常见错误 3：在 Dataset 中直接 `.to("cuda")`

一般不推荐在：

```python
__getitem__()
```

中这样：

```python
image = image.to("cuda")
```

更常见的职责划分是：

```text
Dataset / DataLoader
↓
CPU 侧读取和预处理

训练循环
↓
.to(device)

GPU
↓
模型计算
```

这样更适合 DataLoader 多进程和整体工程管理。

# 28. 常见错误 4：训练集和验证集都使用随机增强

训练集：

```python
RandomCrop
RandomHorizontalFlip
ColorJitter
...
```

没问题。

验证 / 测试集应该尽量：

> **确定、稳定、可重复。**

否则同一个模型每次验证可能因为随机增强得到不同结果。

所以验证通常：

```python
Resize
CenterCrop
ToDtype
Normalize
```

而不是一堆随机增强。

# 29. 常见错误 5：`shuffle=True` 和 `sampler=` 混用

普通训练：

```python
DataLoader(
    dataset,
    shuffle=True
)
```

如果已经指定：

```python
sampler=my_sampler
```

那么样本顺序由 sampler 决定。

因此不要再机械地同时写：

```python
shuffle=True
```

例如：

```python
WeightedRandomSampler
```

场景下通常：

```python
DataLoader(
    dataset,
    sampler=sampler,
    shuffle=False
)
```

甚至直接省略 `shuffle`。

# 30. 完整模板 A：Torchvision 自带分类数据集

```python
from torchvision import datasets
from torchvision.transforms import v2
from torch.utils.data import DataLoader

train_transform = v2.Compose([
    v2.ToImage(),
    v2.RandomHorizontalFlip(),
    v2.ToDtype(torch.float32, scale=True),
])

test_transform = v2.Compose([
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True),
])

train_dataset = datasets.FashionMNIST(
    root="data",
    train=True,
    download=True,
    transform=train_transform,
)

test_dataset = datasets.FashionMNIST(
    root="data",
    train=False,
    download=True,
    transform=test_transform,
)

train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=64,
    shuffle=False,
)
```

# 31. 完整模板 B：自己的文件夹图片分类

```text
data/
├── train/
│   ├── class_a/
│   └── class_b/
└── test/
    ├── class_a/
    └── class_b/
```

```python
train_dataset = datasets.ImageFolder(
    "data/train",
    transform=train_transform,
)

test_dataset = datasets.ImageFolder(
    "data/test",
    transform=test_transform,
)

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False,
)
```

**这是普通自有图像分类项目优先考虑的方法。**

# 32. 完整模板 C：自己的 Tensor / 表格数据

```python
X = torch.tensor(..., dtype=torch.float32)
y = torch.tensor(..., dtype=torch.long)

dataset = TensorDataset(X, y)

train_loader = DataLoader(
    dataset,
    batch_size=64,
    shuffle=True,
)
```

如果数据来自 CSV：

```text
pandas
↓
NumPy
↓
torch.from_numpy
↓
TensorDataset
↓
DataLoader
```

# 33. 完整模板 D：任何复杂自有数据

只要现成 Dataset 不合适：

```python
class MyDataset(Dataset):
    def __init__(self, ...):
        ...

    def __len__(self):
        return ...

    def __getitem__(self, idx):
        ...
        return x, y


dataset = MyDataset(...)

loader = DataLoader(
    dataset,
    batch_size=32,
    shuffle=True,
)
```

### 你真正需要掌握的不是背很多 Dataset 类

而是理解：

```text
任何原始数据
↓
想办法定义 dataset[i]
↓
让它返回一个训练样本
↓
DataLoader 自动组成 batch
```

只要这个思路清楚，绝大多数自定义数据都能处理。

# 34. 选择 Dataset 的决策树

```text
我现在有一批数据
│
├─ PyTorch / Torchvision 已经内置？
│      └─ 是 → datasets.xxx
│
├─ 图片分类，并且“一个类别一个文件夹”？
│      └─ 是 → ImageFolder
│
├─ 数据已经是规则 Tensor / NumPy？
│      └─ 是 → TensorDataset
│
├─ CSV/Excel 且全部能方便放入内存？
│      └─ 是 → pandas → Tensor → TensorDataset
│
├─ 样本读取规则比较特殊？
│      └─ 是 → 自定义 Dataset
│
├─ 每个样本长度/目标数量不同？
│      └─ Dataset + collate_fn
│
└─ 数据太大，只能流式读取？
       └─ IterableDataset
```

# 35. `Dataset → DataLoader → Training Loop` 全流程

最终你应该把数据处理理解成下面这一条线：

```text
原始文件 / 数组
        ↓
      Dataset
        ↓
    Transform
        ↓
     Sampler
        ↓
    DataLoader
        ↓
 ┌───────────────────┐
 │ X_batch, y_batch  │
 └───────────────────┘
        ↓
     .to(device)
        ↓
       model
        ↓
       loss
        ↓
     backward
```

其中：

### Dataset

解决：

> 单个样本是什么？

### Transform

解决：

> 单个样本怎么预处理 / 增强？

### Sampler

解决：

> 样本按什么策略被选出来？

### DataLoader

解决：

> 怎么组成 batch、打乱、多进程加载？

### Training Loop

解决：

> batch 怎么送进模型训练？

# 36. 快速查阅：常用 API 清单

## Dataset 类

```python
torch.utils.data.Dataset
torch.utils.data.IterableDataset
torch.utils.data.TensorDataset
torch.utils.data.Subset
torch.utils.data.ConcatDataset
```

## 划分

```python
torch.utils.data.random_split
```

## DataLoader

```python
torch.utils.data.DataLoader
```

## Sampler

```python
torch.utils.data.WeightedRandomSampler
```

## Torchvision

```python
torchvision.datasets.ImageFolder
torchvision.datasets.FashionMNIST
torchvision.datasets.CIFAR10
...
```

## Transform

```python
from torchvision.transforms import v2
```

常用：

```python
v2.Compose
v2.ToImage
v2.ToDtype
v2.Resize
v2.RandomCrop
v2.RandomResizedCrop
v2.RandomHorizontalFlip
v2.RandomRotation
v2.ColorJitter
v2.Normalize
v2.RandomErasing
```

# 37. 最后只背这 8 条

如果以后时间很紧，只复习这里：

### ① Dataset 管单个样本

```python
dataset[i]
```

---

### ② DataLoader 管 batch

```python
for X, y in loader:
    ...
```

---

### ③ 官方数据集

```python
datasets.FashionMNIST(...)
```

---

### ④ 图片按类别文件夹放

```python
datasets.ImageFolder(...)
```

---

### ⑤ 已有 Tensor

```python
TensorDataset(X, y)
```

---

### ⑥ 特殊数据

```python
class MyDataset(Dataset):
    def __len__(self):
        ...

    def __getitem__(self, idx):
        ...
```

---

### ⑦ 样本长度不同

```python
collate_fn
```

---

### ⑧ 超大 / 流式数据

```python
IterableDataset
```

最终最重要的一句话：

> **PyTorch 数据处理的本质，就是把任何形式的原始数据包装成 Dataset，再由 DataLoader 按 batch 提供给训练循环。**

# 38. 自测：确认自己真正会用了

建议复习完后直接回答：

1. `Dataset` 和 `DataLoader` 的核心区别是什么？
2. 为什么 `FashionMNIST` 不需要自己实现 `Dataset`？
3. 什么目录结构最适合 `ImageFolder`？
4. 数据已经是 `X`、`y` 两个 Tensor 时用什么？
5. CSV 表格数据一定要自定义 Dataset 吗？
6. `__getitem__(self, idx)` 到底负责什么？
7. 为什么大图片数据通常不在 `__init__` 中全部读入内存？
8. 图像分割为什么 image 和 mask 必须同步随机增强？
9. 什么情况下需要 `collate_fn`？
10. 为什么变长文本默认不能直接 `torch.stack`？
11. `IterableDataset` 适合什么数据？
12. 训练集为什么通常 `shuffle=True`，测试集通常 `False`？
13. `sampler` 和 `shuffle` 是什么关系？
14. `pin_memory` 一般和什么硬件场景有关？
15. Dataset 调试时为什么先检查 `dataset[0]`，而不是直接训练？

# 39. 官方参考（建议忘记细节时优先查）

本笔记按 PyTorch / Torchvision 当前官方文档整理，重点参考：

- PyTorch — `torch.utils.data`  
  https://docs.pytorch.org/docs/stable/data.html

- PyTorch Tutorial — Datasets & DataLoaders  
  https://docs.pytorch.org/tutorials/beginner/basics/data_tutorial.html

- Torchvision — Datasets  
  https://docs.pytorch.org/vision/stable/datasets.html

- Torchvision — Transforms v2  
  https://docs.pytorch.org/vision/main/transforms.html

- PyTorch Tutorial — Data Loading Optimization  
  https://docs.pytorch.org/tutorials/intermediate/intermediate_data_loading_tutorial.html

> 注：Torchvision 当前官方推荐优先使用 `torchvision.transforms.v2`。阅读较老教材或项目时仍会频繁看到 `torchvision.transforms` / `ToTensor()`，理解两套写法的对应关系即可。